In [1]:
# from datasets import load_dataset, concatenate_datasets

# df = load_dataset("Qwen/ProcessBench")
# df = concatenate_datasets(df.values()).to_pandas()

# df["split"] = df["id"].str.split("-").str[0]
# df["steps_len"] = df["steps"].str.len()
# df["per_step_len"] = df["steps"].apply(lambda x: [len(y) for y in x])

import pandas as pd

# df = pd.read_parquet("data/processbench1000_length_llm.parquet")
df = pd.read_parquet("data/processbench_with_roi2.parquet")
df.keys()

Index(['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label',
       'split', 'steps_len', 'per_step_len', 'Qwen2.5-Math-PRM-7B',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B', 'roi_step_num', 'roi_step',
       'verbose_roi_step', 'consise_roi_step', 'spelled_eq_roi_step',
       'verbose_steps', 'consise_steps', 'spelled_eq_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--spelled_eq_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--consise_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--verbose_steps',
       'Qwen2.5-Math-PRM-7B--verbose_steps',
       'Qwen2.5-Math-PRM-7B--consise_steps',
       'Qwen2.5-Math-PRM-7B--spelled_eq_steps', 'change_numbers_roi_step',
       'change_numbers_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--change_numbers_steps',
       'Qwen2.5-Math-PRM-7B--change_numbers_steps', 'verbose_roi_step_v2',
       'verbose_steps_v2',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--verbose_steps_v2'],
      dtype='object')

In [2]:
## Sanity check
row = df.sample().iloc[0]
print(row["label"], row["roi_step_num"])
print("### Steps ###")
print(row["steps"])

print("### Verbose ###")
print(row["verbose_steps_v2"])

print("### Consise ###")
print(row["consise_steps"])

print("### Spelled EQ ###")
print(row["spelled_eq_steps"])

-1 0
### Steps ###
["To solve the equation \\(2^{11} \\times 6^{5} = 4^{x} \\times 3^{y}\\), let's first express all terms in terms of prime factors to simplify the equation. Given:\n\\[2^{11} \\times 6^{5} = 4^{x} \\times 3^{y}\\]"
 'Firstly, note that \\(6 = 2 \\times 3\\), so we can rewrite \\(6^5\\) as:\n\\[6^{5} = (2 \\times 3)^{5} = 2^{5} \\times 3^{5}\\]'
 'And, \\(4 = 2^2\\), so \\(4^x\\) can be rewritten as:\n\\[4^{x} = (2^2)^{x} = 2^{2x}\\]'
 'Substituting these into our original equation gives:\n\\[2^{11} \\times 2^{5} \\times 3^{5} = 2^{2x} \\times 3^{y}\\]'
 'Combining like bases on the left side, we get:\n\\[2^{11+5} \\times 3^{5} = 2^{2x} \\times 3^{y}\\]\n\\[2^{16} \\times 3^{5} = 2^{2x} \\times 3^{y}\\]'
 'From this equation, we can equate the exponents of the same bases:\nFor base 2: \\(16 = 2x\\)\nFor base 3: \\(5 = y\\)'
 'Solving for \\(x\\) from \\(16 = 2x\\) gives:\n\\[x = \\frac{16}{2} = 8\\]'
 'And since \\(y = 5\\) directly from the equation for base 3, we hav

In [ ]:
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer
from utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

```bash
vllm serve hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
vllm serve hf_cache/Qwen--Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [4]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8081/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
models = client.models.list()
model = models.data[0].id

In [5]:
print(model)
tokenizer = AutoTokenizer.from_pretrained(model)

hf_cache/Qwen--Qwen2.5-Math-PRM-7B


In [6]:
batch_size = 4
num_samples = len(df)

# for steps_col in ["spelled_eq_steps", "consise_steps", "verbose_steps", "change_numbers_steps"]:
for steps_col in ["verbose_steps_v2"]:
    input_ids_all = []
    token_mask_all = []
    all_rewards = []

    for data in tqdm(df.T.to_dict().values(), total=len(df)):
        input_ids, token_mask = prepare_input(
                                model, 
                                problem=data["problem"], 
                                steps=data[steps_col], 
                                tokenizer=tokenizer,
                                convert_to_list=True
        )
        input_ids_all.append(input_ids)
        token_mask_all.append(token_mask)


    for start_idx in tqdm(range(0, num_samples, batch_size), desc=steps_col):
        end_idx = start_idx + batch_size
    
        batch_input_ids = input_ids_all[start_idx:end_idx]
        batch_token_masks = token_mask_all[start_idx:end_idx]
    
        batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)
    
        batch_logits = client.embeddings.create(
            input=batch_input_ids.cpu().tolist(),
            model=model,
        )
    
        rewards = derive_step_rewards_vllm(
            model,
            batch_logits,
            batch_token_masks,
            tokenizer
        )
    
        all_rewards.extend(rewards)

    df[f"{model.split('--')[-1]}--{steps_col}"] = all_rewards
    df.to_parquet("data/processbench_with_roi2.parquet", index=False)

  0%|          | 0/3400 [00:00<?, ?it/s]

verbose_steps_v2:   0%|          | 0/850 [00:00<?, ?it/s]

In [7]:
df

,id,generator,problem,steps,final_answer_correct,label,split,steps_len,per_step_len,Qwen2.5-Math-PRM-7B,...,Qwen2.5-Math-PRM-7B--consise_steps,Qwen2.5-Math-PRM-7B--spelled_eq_steps,change_numbers_roi_step,change_numbers_steps,Skywork-o1-Open-PRM-Qwen-2.5-7B--change_numbers_steps,Qwen2.5-Math-PRM-7B--change_numbers_steps,verbose_roi_step_v2,verbose_steps_v2,Skywork-o1-Open-PRM-Qwen-2.5-7B--verbose_steps_v2,Qwen2.5-Math-PRM-7B--verbose_steps_v2
0,gsm8k-0,Qwen2-7B-Instruct,"Sue lives in a fun neighborhood. One weekend,...",[To find out how many more pink plastic flamin...,False,1,gsm8k,4,"[217, 447, 182, 250]","[1.0, 0.1865234375, 0.9765625, 1.0]",...,"[1.0, 0.55078125, 0.97265625, 1.0]","[1.0, 0.10009765625, 0.97265625, 1.0]","On Saturday, they take back one third of the f...",[To find out how many more pink plastic flamin...,"[0.7534666539902642, 0.09947021388210706, 0.15...","[1.0, 0.00897216796875, 0.0164794921875, 0.341...","On Saturday, they take back one third of the f...",[To find out how many more pink plastic flamin...,"[0.7534666539902642, 0.5893618976069925, 0.486...","[1.0, 0.72265625, 0.7421875, 1.0]"
1,gsm8k-1,Qwen2.5-Math-7B-Instruct,Cindy's math and science books weigh 2 pounds ...,[To determine the total weight of all Cindy's ...,False,1,gsm8k,7,"[137, 461, 63, 64, 170, 249, 84]","[1.0, 0.031494140625, 0.99609375, 1.0, 0.99609...",...,"[1.0, 0.0081787109375, 0.953125, 0.99609375, 0...","[1.0, 0.2109375, 0.99609375, 1.0, 0.98828125, ...",Each math book weighs 3 pounds.\n- Each scienc...,[To determine the total weight of all Cindy's ...,"[0.9579122720843811, 0.024053552894426174, 0.0...","[1.0, 0.007110595703125, 0.96484375, 0.9960937...","For the math and science books, each math book...",[To determine the total weight of all Cindy's ...,"[0.9572778024535095, 0.052618953347713764, 0.0...","[1.0, 0.07861328125, 0.99609375, 1.0, 0.992187..."
2,gsm8k-2,Qwen2-7B-Instruct,A company sold 4000 gallons of milk in jars to...,"[First, let's calculate the total cost of the ...",False,1,gsm8k,4,"[212, 207, 168, 102]","[1.0, 0.248046875, 0.99609375, 1.0]",...,"[1.0, 0.05078125, 0.98046875, 1.0]","[1.0, 0.0091552734375, 0.09814453125, 0.94140625]",Fraction of expired milk = 3/5. Amount of expi...,"[First, let's calculate the total cost of the ...","[0.8799743722456386, 0.048857778622174906, 0.0...","[1.0, 0.00159454345703125, 0.03369140625, 0.39...","Next, we need to determine the quantity of mil...","[First, let's calculate the total cost of the ...","[0.8749346174830964, 0.5583269943353745, 0.763...","[1.0, 0.058349609375, 0.98046875, 1.0]"
3,gsm8k-3,Qwen2-1.5B-Instruct,A class of 50 students has various hobbies. 10...,[To find out how many students like to play vi...,False,2,gsm8k,4,"[296, 129, 519, 55]","[1.0, 1.0, 0.0693359375, 0.1298828125]",...,"[1.0, 1.0, 0.048095703125, 0.054931640625]","[1.0, 1.0, 0.01953125, 0.267578125]",\[ \text{Number of students who like to play v...,[To find out how many students like to play vi...,"[0.8791467675095294, 0.8902942261220291, 0.033...","[1.0, 1.0, 0.0031280517578125, 0.01019287109375]","Now, we subtract the number of students who en...",[To find out how many students like to play vi...,"[0.8824278664544911, 0.8902942261220291, 0.088...","[1.0, 1.0, 0.08447265625, 0.11181640625]"
4,gsm8k-4,Qwen2-7B-Instruct,A customer’s loyalty card at a store gives the...,"[Firstly, let's break down the problem into pa...",False,2,gsm8k,5,"[75, 227, 418, 216, 55]","[0.99609375, 0.99609375, 0.00360107421875, 0.6...",...,"[0.99609375, 0.99609375, 0.03369140625, 0.2412...","[0.99609375, 0.99609375, 0.01416015625, 0.4414...",The current shopping trip involved spending $5...,"[Firstly, let's break down the problem into pa...","[0.6233768743729451, 0.7505527565366589, 0.050...","[0.99609375, 0.99609375, 0.00262451171875, 0.0...",The current shopping trip involved spending $4...,"[Firstly, let's break down the problem into pa...","[0.625209303080424, 0.7520125737597972, 0.0685...","[0.99609375, 0.9960937

---

In [233]:
row = df.iloc[243] # QWEN 1944

In [234]:
row.problem

'If $m$ is a real number and $2x^2+mx+8$ has two distinct real roots, then what are the possible values of $m$?  Express your answer in interval notation.'

In [235]:
row.roi_step_num

0

In [236]:
row.label

0

In [237]:
row["consise_steps"]

array(['Since $2x^2+mx+8$ has two distinct real roots, the discriminant $m^2-64>0$, leading to $(m-8)(m+8)>0$.',
       'This gives us that either $m-4>0$ and $m+8>0$, or $m-4<0$ and $m+8<0$. The first set of inequalities can be simplified to $m>4$ and $m>-8$. Since $m>4$ already implies $m>-8$, the solution for this case is $m>4$.',
       'The second set of inequalities can be simplified to $m<4$ and $m<-8$. Since $m<-8$ already implies $m<4$, the solution for this case is $m<-8$.',
       'Combining both cases, the possible values of $m$ are $m>4$ or $m<-8$. Therefore, the solution in interval notation is $\\boxed{(-\\infty, -8) \\cup (4, \\infty)}$. Final Answer: The final answer is $(-\\infty, -8) \\cup (4, \\infty)$. I hope it is correct.'],
      dtype=object)

In [238]:
row.steps

array(['Since $2x^2+mx+8$ has two distinct real roots, its discriminant must be positive. The discriminant is given by $b^2-4ac$, where $a=2$, $b=m$, and $c=8$. Thus, we have that $m^2-4(2)(8)>0$, so $m^2-32>0$, which implies that $(m-4)(m+8)>0$.',
       'This gives us that either $m-4>0$ and $m+8>0$, or $m-4<0$ and $m+8<0$. The first set of inequalities can be simplified to $m>4$ and $m>-8$. Since $m>4$ already implies $m>-8$, the solution for this case is $m>4$.',
       'The second set of inequalities can be simplified to $m<4$ and $m<-8$. Since $m<-8$ already implies $m<4$, the solution for this case is $m<-8$.',
       'Combining both cases, the possible values of $m$ are $m>4$ or $m<-8$. Therefore, the solution in interval notation is $\\boxed{(-\\infty, -8) \\cup (4, \\infty)}$. Final Answer: The final answer is $(-\\infty, -8) \\cup (4, \\infty)$. I hope it is correct.'],
      dtype=object)

In [227]:
my_steps = row.steps


In [228]:
input_ids, token_mask = prepare_input(
                                model, 
                                problem=row.problem, 
                                steps=my_steps, 
                                tokenizer=tokenizer,
                                convert_to_list=True
                        )

In [229]:
print(tokenizer.decode(input_ids))

<|im_start|>An integer $N$ is worth 1 point for each pair of digits it contains that forms a prime in its original order. For example, 6733 is worth 3 points (for 67,73 , and 73 again), and 20304 is worth 2 points (for 23 and 03). Compute the smallest positive integer that is worth exactly 11 points. [Note: Leading zeros are not allowed in the original integer.]
Tofindthesmallestpositiveintegerthatisworthexactly11points,weneedtoidentifypairsofdigitsthatformprimenumbersandensurethetotalcountofsuchpairsis11.Wewillstartbylistingalltwo-digitprimenumbers:11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,83,89,97.Weaimtoconstructthesmallestintegerbyusingtheseprimesaspairsofdigits.Tominimizethenumber,weshouldusethesmallestpossibledigits.
First,identifythesmallestprimes:Thesmallestprimesare11,13,17,19,23,29,31,37,41,43,47.Wewillusetheseprimestoformournumber.
Second,formthesmallestnumberwith11primes:-Startwiththesmallestprimesandtrytooverlapthemtominimizethenumberofdigits.-Forexample,11and1

In [230]:
logits = client.embeddings.create(
        input=input_ids,
        model=model,
    )

In [231]:
import torch
derive_step_rewards_vllm(
    model,
    logits,
    torch.Tensor([token_mask]),
    tokenizer
)#[0][3]

[[0.035678550877730524,
  0.037892426629607394,
  0.05582314113461443,
  0.19806417366892506,
  0.13296424019782926]]